In [1]:
import sys
from pathlib import Path
from functools import partial
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from typing import Optional
from omegaconf import OmegaConf

import numpy as np
import torch
from torchtune import config
import torch.nn.functional as F


from torch.utils.data import DataLoader, Dataset
from torchtune.data import padded_collate_packed
from torchtune.data._common import CROSS_ENTROPY_IGNORE_IDX, PACK_TYPE
from torchtune.modules.transforms.tokenizers import BaseTokenizer, ModelTokenizer
from dataset_classes.stochastic_languages import PFADataset
from torchtune.data._utils import truncate

cfg = OmegaConf.load(f'{PROJECT_ROOT}/configs/llama_0.1B_PHi.yaml')

cfg.dataset._component_ = 'learning_levels_pfa_dataset'
cfg.dataset.max_sample_length = 512

/home/woody/iwbi/iwbi106h/software/private/conda/envs/hsp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = config.instantiate(cfg.tokenizer)

In [3]:
class PackedOnTheFlyDataset(torch.utils.data.IterableDataset):
    """
    An IterableDataset that packs sequences from an underlying dataset on-the-fly.

    This dataset takes an existing PyTorch Dataset and iterates through it,
    concatenating sequences together until a `max_seq_len` is reached.
    It handles splitting individual samples across multiple packs if necessary
    (controlled by `split_across_pack`) and can process data in a distributed
    manner across multiple workers and ranks. It also supports permuting the
    order of samples from the underlying dataset.

    The output of each iteration is a dictionary (pack) containing 'tokens',
    'labels', 'input_pos', 'seq_lens', and any other sequences present in the
    original dataset, all padded to `max_seq_len`. It also includes metadata like
    '_debug_example_idx' to trace back to original samples.
    """
    def __init__(
        self,
        ds: Dataset,
        *,
        max_seq_len: int,
        padding_idx: int = 0,
        max_packs: Optional[int] = None,
        split_across_pack: bool = False,
        permute_indices: bool = False,
        world_size: int = 1,
        rank: int = 0,
        verbose: int = 0,
    ):
        super(PackedOnTheFlyDataset).__init__()
        self.ds = ds
        self.world_size = world_size
        self.rank = rank
        self.verbose = verbose
        if self.world_size > 1:
            start_idx = int(rank * len(self.ds) / world_size)
            end_idx = int((rank + 1) * len(self.ds) / world_size)
            if self.verbose > 0:
                print(
                    f"Rank {rank} trains on indices {start_idx} to {end_idx} (out of {len(self.ds)})"
                )
            self.ds = Subset(self.ds, range(start_idx, end_idx))
        self.max_seq_len = max_seq_len
        self.padding_idx = padding_idx
        self.max_packs = max_packs
        self.split_across_pack = split_across_pack
        self.current_idx = 0
        self.current_sample = (
            None  # if we need to split a sample across packs, it will be stored here
        )
        self.current_sample_cutoff = 0  # if we need to split a sample across packs, this will store the last sequence cutoff
        self.end_idx = len(self.ds)
        self.permute_indices = permute_indices
        if self.permute_indices:
            self.dataset_idxs = torch.randperm(len(self.ds))
        else:
            self.dataset_idxs = torch.arange(len(self.ds))

    @staticmethod
    def _worker_init_fn(worker_id, verbose=0):
        worker_info = torch.utils.data.get_worker_info()
        dataset = worker_info.dataset

        dataset.current_idx = int(
            worker_info.id * len(dataset.dataset_idxs) / worker_info.num_workers
        )
        dataset.end_idx = int(
            (worker_info.id + 1) * len(dataset.dataset_idxs) / worker_info.num_workers
        )
        if dataset.permute_indices and verbose > 0:
            print(
                f"worker {worker_id} is going through indices {dataset.current_idx} to "
                f"{dataset.end_idx} of permutation {dataset.dataset_idxs[:5]})"
            )
            print(
                f"permutation indices in worker {worker_id}: {dataset.dataset_idxs[dataset.current_idx:dataset.current_idx + 5]}..."
            )

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_idx >= self.end_idx:
            raise StopIteration

        num_tokens = 0
        num_packs = 0
        current_pack = {
            "input_pos": [],
            "_seq_lens": [],
            "_debug_example_idx": [],
        }
        while num_tokens < self.max_seq_len:
            if self.current_idx >= self.end_idx:
                break

            if self.max_packs is not None and num_packs >= self.max_packs:
                break

            if self.current_sample is None:
                self.current_sample = self.ds[
                    self.dataset_idxs[self.current_idx].item()
                ]

            # Dynamically discover sequence and meta-information keys
            keys = self.current_sample.keys()
            is_meta_information = [type(self.current_sample[key]) != list for key in keys]
            meta_information_keys = [key for key, is_meta in zip(keys, is_meta_information) if is_meta]
            sequence_keys = [key for key in keys if key not in meta_information_keys]

            values = [self.current_sample[key] for key in sequence_keys]
            meta_information_values = [self.current_sample[key] for key in meta_information_keys]
            meta_information_keys = ['_' + key for key in meta_information_keys]
            seq_len = len(values[0])
            for key in sequence_keys:
                if key not in current_pack:
                    current_pack[key] = []
            for key in meta_information_keys:
                if key not in current_pack:
                    current_pack[key] = []

            if num_tokens + seq_len <= self.max_seq_len:
                # Add entire (remaining) sample to pack
                for key, value in zip(sequence_keys, values):
                    current_pack[key] += value
                for key, value in zip(meta_information_keys, meta_information_values):
                    current_pack[key] += [value]

                current_pack["input_pos"] += list(range(seq_len))
                current_pack["_seq_lens"] += [seq_len]
                current_pack["_debug_example_idx"] += [
                    self.dataset_idxs[self.current_idx].item()
                ]
                num_tokens += seq_len
                self.current_sample_cutoff = 0
                self.current_idx += 1
                num_packs += 1
                self.current_sample = None
            elif self.split_across_pack or num_tokens == 0:
                # Split sample or add first part if pack is empty
                remaining_sequence_len = self.max_seq_len - num_tokens
                for key, value in zip(sequence_keys, values):
                    current_pack[key] += value[:remaining_sequence_len]
                for key, value in zip(meta_information_keys, meta_information_values):
                    current_pack[key] += [value]

                current_pack["input_pos"] += list(range(remaining_sequence_len))
                current_pack["_seq_lens"] += [remaining_sequence_len]
                current_pack["_debug_example_idx"] += [
                    self.dataset_idxs[self.current_idx].item()
                ]
                num_tokens += remaining_sequence_len
                self.current_sample_cutoff += remaining_sequence_len
                break # Pack is full
            else:
                break # Pack is full

        pack = self._convert_to_tensors(current_pack)
        pack = self._pad_pack(pack, padding_idx=self.padding_idx)
        pack["seq_lens"] = torch.tensor(pack["_seq_lens"], dtype=torch.long)
        del pack["_seq_lens"]
        return pack

    def _convert_to_tensors(self, pack: PACK_TYPE) -> PACK_TYPE:
        """Converts a pack into tensors. Pack comes in as a dict of lists and is converted to tensors."""
        return_dict = {
            key: torch.tensor(value, dtype=torch.long) if key[0] != "_" else value for key, value in pack.items()
        }
        return return_dict

    def _pad_pack(self, pack: PACK_TYPE, padding_idx: int) -> PACK_TYPE:
        """Pads a pack to ``self.max_seq_len``."""
        # Pad tokens
        num_padding_tokens = self.max_seq_len - len(pack["tokens"])
        padded_tokens = F.pad(
            pack["tokens"],
            (0, num_padding_tokens),
            value=padding_idx,
        )

        # Pad labels
        padded_labels = F.pad(
            pack["labels"],
            (0, self.max_seq_len - len(pack["labels"])),
            value=CROSS_ENTROPY_IGNORE_IDX,
        )

        # Add padding tokens as a last seq len to ensure sum is max_seq_len
        padded_seq_lens = pack["_seq_lens"] + [num_padding_tokens] if num_padding_tokens > 0 else pack["_seq_lens"]

        # Pad debug dataset idx
        padded_debug_example_idx = pack["_debug_example_idx"]

        # Pad input_pos continuing the sequence from last value
        # in input_pos
        # e.g. [0 1 2] -> [0 1 2 3 4 5] for self.max_seq_len = 6
        num_range = torch.arange(
            pack["input_pos"][-1] + 1,
            pack["input_pos"][-1] + self.max_seq_len - len(pack["input_pos"]) + 1,
        )
        # Clamp to max_seq_len - 1 to avoid out of bounds error
        clamped_num_range = torch.clamp(num_range, 0, self.max_seq_len - 1)
        padded_input_pos = torch.cat([pack["input_pos"], clamped_num_range])

        return_dict = {
            "tokens": padded_tokens,
            "labels": padded_labels,
            "input_pos": padded_input_pos,
            "_seq_lens": padded_seq_lens,
            "_debug_example_idx": padded_debug_example_idx,
        }

        # account for all additional sequences
        for key in pack.keys():
            if key in ["tokens", "labels", "input_pos", "_seq_lens"]:
                continue

            if key[0] == "_":
                return_dict[key] = pack[key]
                continue

            padded = F.pad(
                pack[key],
                (0, self.max_seq_len - len(pack[key])),
                value=padding_idx,
            )
            return_dict[key] = padded

        return return_dict


In [4]:
def learning_levels_pfa_dataset(
    tokenizer: BaseTokenizer,
    num_states_min: int = 4,
    num_states_max: int = 12,
    alphabet_size_min: int = 4,
    alphabet_size_max: int = 18,
    seq_len_min: int = 1,
    seq_len_max: int = 50,
    edges_per_state_min: int = 1,
    edges_per_state_max: int = 4,
    sequences_per_language_min: int = 10,
    sequences_per_language_max: int = 20,
    max_sample_length: int = 2048,
    num_fixed_automata: int = 10,
    num_fixed_sequences: int = 10,
    fixed_seed: int = 123,
    word_perturbation_rate: float = 0.5,
    token_perturbation_rate: float = 0.2,
    included_learning_levels=(0, 1, 2, 3, 4, 5),
    max_num_languages_per_sample: Optional[int] = None,
    constrained_sequences: int = 0,
    edge_constrain_ratio: float = 0.2,
    shuffle_random_sequences: bool = True,
):
    """
    Factory function to create a `LearningLevelsPFADataset`.

    This function initializes and returns an instance of `LearningLevelsPFADataset`,
    which generates complex sequences based on Probabilistic Finite Automata (PFAs)
    across multiple, distinct levels of learning complexity. All arguments are
    passed directly to the `LearningLevelsPFADataset` constructor.

    Args:
        tokenizer (BaseTokenizer): Tokenizer for encoding generated sequences.
        num_states_min (int, optional): Min states in a PFA. Defaults to 4.
        num_states_max (int, optional): Max states in a PFA. Defaults to 12.
        alphabet_size_min (int, optional): Min vocabulary size for a PFA. Defaults to 4.
        alphabet_size_max (int, optional): Max vocabulary size for a PFA. Defaults to 18.
        seq_len_min (int, optional): Min length of a single sequence. Defaults to 1.
        seq_len_max (int, optional): Max length of a single sequence. Defaults to 50.
        edges_per_state_min (int, optional): Min outgoing edges from a state. Defaults to 1.
        edges_per_state_max (int, optional): Max outgoing edges from a state. Defaults to 4.
        sequences_per_language_min (int, optional): Min sequences per language block. Defaults to 10.
        sequences_per_language_max (int, optional): Max sequences per language block. Defaults to 20.
        max_sample_length (int, optional): Target character length for a full sample. Defaults to 2048.
        num_fixed_automata (int, optional): Number of fixed PFAs for levels 0, 1, and 2. Defaults to 10.
        num_fixed_sequences (int, optional): Number of fixed sequences per PFA for level 0. Defaults to 10.
        fixed_seed (int, optional): Seed for generating fixed automata and sequences. Defaults to 123.
        word_perturbation_rate (float, optional): Probability of perturbing a sequence. Defaults to 0.5.
        token_perturbation_rate (float, optional): Probability of perturbing a token in a perturbed sequence. Defaults to 0.2.
        included_learning_levels (Tuple[int, ...], optional): Which learning levels (0-5) to sample from.
            Defaults to (0, 1, 2, 3, 4, 5).
        max_num_languages_per_sample (Optional[int], optional): Max number of language blocks per sample.
            Defaults to None (unlimited).
        constrained_sequences (int, optional): Number of initial sequences in a block to generate from
            a more predictable PFA. Defaults to 0.
        edge_constrain_ratio (float, optional): Ratio of PFA edges to disable for constrained generation.
            Defaults to 0.2.
        shuffle_random_sequences (bool, optional): For the copying task (level 5), whether to shuffle
            the repeated sequences. Defaults to True.

    Returns:
        LearningLevelsPFADataset: An instance of the configured dataset.
    """
    ds = LearningLevelsPFADataset(
        tokenizer=tokenizer,
        num_states_min=num_states_min,
        num_states_max=num_states_max,
        alphabet_size_min=alphabet_size_min,
        alphabet_size_max=alphabet_size_max,
        seq_len_min=seq_len_min,
        seq_len_max=seq_len_max,
        edges_per_state_min=edges_per_state_min,
        edges_per_state_max=edges_per_state_max,
        sequences_per_language_min=sequences_per_language_min,
        sequences_per_language_max=sequences_per_language_max,
        max_sample_length=max_sample_length,
        num_fixed_automata=num_fixed_automata,
        num_fixed_sequences=num_fixed_sequences,
        fixed_seed=fixed_seed,
        word_perturbation_rate=word_perturbation_rate,
        token_perturbation_rate=token_perturbation_rate,
        included_learning_levels=included_learning_levels,
        max_num_languages_per_sample=max_num_languages_per_sample,
        constrained_sequences=constrained_sequences,
        edge_constrain_ratio=edge_constrain_ratio,
        shuffle_random_sequences=shuffle_random_sequences,
    )
    return ds


In [5]:
class LearningLevelsPFADataset(PFADataset):
    """
    Generates sequences from Probabilistic Finite Automata (PFAs) across multiple,
    distinct levels of learning complexity.

    This dataset is designed for studying in-context learning by creating tasks that
    range from simple memorization to complex, on-the-fly inference. It extends
    `PFADataset` by defining several "learning levels," each corresponding to a
    different type of sequence generation task. When an item is requested, it
    constructs a sample by randomly mixing blocks of sequences from these different levels.

    The defined learning levels are:
    - **Level 0 (Memorized Sequences):** Retrieves pre-generated, fixed sequences from a
      fixed set of automata.
    - **Level 1 (Memorized Programs):** Generates new sequences on-the-fly, but from a
      fixed, pre-generated set of automata.
    - **Level 2 (Fixed Structure, New Vocab):** Uses pre-generated automaton structures
      but swaps their character vocabularies at generation time.
    - **Level 3 (In-Context Learning):** Generates sequences from entirely new, random
      automata on-the-fly.
    - **Level 4 (Random):** Generates sequences of completely random characters, with no
      underlying structure.
    - **Level 5 (Copying):** Generates random sequences and then repeats them, creating a
      copying task.

    Args:
        tokenizer (ModelTokenizer, optional): Tokenizer for encoding the generated sequences.
            If None, raw character strings are returned. Defaults to None.
        num_states_min (int, optional): Min states in a generated PFA. Defaults to 4.
        num_states_max (int, optional): Max states in a generated PFA. Defaults to 12.
        alphabet_size_min (int, optional): Min vocabulary size for a PFA. Defaults to 4.
        alphabet_size_max (int, optional): Max vocabulary size for a PFA. Defaults to 18.
        seq_len_min (int, optional): Min length of a single generated sequence. Defaults to 1.
        seq_len_max (int, optional): Max length of a single generated sequence. Defaults to 50.
        edges_per_state_min (int, optional): Min outgoing edges from a PFA state. Defaults to 1.
        edges_per_state_max (int, optional): Max outgoing edges from a PFA state. Defaults to 4.
        sequences_per_language_min (int, optional): Min sequences per language block. Defaults to 10.
        sequences_per_language_max (int, optional): Max sequences per language block. Defaults to 20.
        max_sample_length (int, optional): Target character length for a full sample. Defaults to 2048.
        num_fixed_automata (int, optional): Number of fixed PFAs to pre-generate for levels
            0, 1, and 2. Defaults to 10.
        num_fixed_sequences (int, optional): Number of fixed sequences to generate per PFA
            for level 0. Defaults to 10.
        fixed_seed (int, optional): A seed for the random number generator to ensure the
            fixed automata and sequences are the same across runs. Defaults to 123.
        shuffle_random_sequences (bool, optional): For level 5, if True, all copied sequences
            are shuffled together. If False, each sequence is repeated immediately. Defaults to True.
        word_perturbation_rate (float, optional): Probability of applying token-level
            perturbations to a generated sequence. Defaults to 0.5.
        token_perturbation_rate (float, optional): If a sequence is perturbed, this is the
            probability that any given token within it will be replaced by a random one. Defaults to 0.2.
        included_learning_levels (tuple, optional): A tuple of integer learning levels (0-5)
            to sample from when generating data. Defaults to (0, 1, 2, 3, 4, 5).
        max_num_languages_per_sample (Optional[int], optional): The maximum number of different
            language/level blocks to include in a single sample. Defaults to None (infinite).
        constrained_sequences (int, optional): If > 0, for this many initial sequences in a PFA
            block, a more predictable (constrained) version of the PFA is used. Defaults to 0.
        edge_constrain_ratio (float, optional): The ratio of edges to temporarily disable to
            create a constrained PFA. Defaults to 0.2.
    """
    def __init__(
        self,
        tokenizer: ModelTokenizer = None,
        num_states_min: int = 4,
        num_states_max: int = 12,
        alphabet_size_min: int = 4,
        alphabet_size_max: int = 18,
        seq_len_min: int = 1,
        seq_len_max: int = 50,
        edges_per_state_min: int = 1,
        edges_per_state_max: int = 4,
        sequences_per_language_min: int = 10,
        sequences_per_language_max: int = 20,
        max_sample_length: int = 2048,
        num_fixed_automata: int = 10,
        num_fixed_sequences: int = 10,
        fixed_seed: int = 123,
        shuffle_random_sequences: bool = True,
        word_perturbation_rate: float = 0.5,
        token_perturbation_rate: float = 0.2,
        included_learning_levels=(0, 1, 2, 3, 4, 5),
        max_num_languages_per_sample: Optional[int] = None,
        constrained_sequences: int = 0,
        edge_constrain_ratio: float = 0.2,
    ):
        super().__init__(
            tokenizer=tokenizer,
            num_states_min=num_states_min,
            num_states_max=num_states_max,
            alphabet_size_min=alphabet_size_min,
            alphabet_size_max=alphabet_size_max,
            seq_len_min=seq_len_min,
            seq_len_max=seq_len_max,
            edges_per_state_min=edges_per_state_min,
            edges_per_state_max=edges_per_state_max,
            sequences_per_language_min=sequences_per_language_min,
            sequences_per_language_max=sequences_per_language_max,
            max_sample_length=max_sample_length,
        )

        self.word_perturbation_rate = word_perturbation_rate
        self.token_perturbation_rate = token_perturbation_rate
        self.included_learning_levels = included_learning_levels
        if max_num_languages_per_sample is None:
            self.max_num_languages_per_sample = float("inf")
        else:
            self.max_num_languages_per_sample = max_num_languages_per_sample
        self.constrained_sequences = constrained_sequences
        self.edge_constrain_ratio = edge_constrain_ratio
        self.shuffle_random_sequences = shuffle_random_sequences

        # There are three levels:
        # - Level 0: fixed sequences from fixed automata
        # - Level 1: generated sequences from fixed automata
        # - Level 2: generated sequences from random automata with fixed structure (different vocabulary)
        # - Level 3: generates sequences from random automata
        # - Level 4: generates completely random sequences
        # - Level 5: generates random sequences and repeats them (in shuffled order)

        # The next part needs a fixed random seed
        seed_state = torch.get_rng_state()
        torch.manual_seed(fixed_seed)

        # Level 0: Generate languages and save fixed sequences from them
        self.level0_languages = [
            self.generate_language() for _ in range(num_fixed_automata)
        ]
        self.level0_sequences = []
        for transition_probs, transition_symbols in self.level0_languages:
            s = []
            for _ in range(num_fixed_sequences):
                sequence, _ = self.generate_sequence(
                    transition_probs, transition_symbols
                )
                s.append(sequence)
            self.level0_sequences.append(s)

        # Level 1: Generate languages
        self.level1_languages = [
            self.generate_language() for _ in range(num_fixed_automata)
        ]

        # Level 2: Generate langauges (to be modified when the sequence is generated)
        self.level2_languages = [
            self.generate_language() for _ in range(num_fixed_automata)
        ]

        # Levels 3, 4 & 5: Everything is generated on-the-fly, nothing to do here

        # Return to initial random state
        torch.set_rng_state(seed_state)

    def __getitem__(self, item):
        """
        Generates and returns a single training sample composed of sequences from
        various learning levels.

        Args:
            item (int): The index of the item (ignored, as data is generated on-the-fly).

        Returns:
            dict: A dictionary containing the tokenized sample and extensive metadata,
                with all values being lists of the same length as the token sequence:
                - "tokens" (List[int]): The tokenized sequence.
                - "labels" (List[int]): A copy of the tokens for language modeling.
                - "new_language" (List[bool]): A boolean flag, True at the start of a new
                  language/level block.
                - "learning_level" (List[int]): An integer (0-5) indicating the complexity
                  level of each token.
                - "num_states" (List[int]): The number of states in the PFA that generated
                  the token (0 for non-PFA levels).
                - "num_edges" (List[int]): The number of edges in the PFA.
                - "vocab_size" (List[int]): The vocabulary size of the PFA.
                - "perturbation" (List[bool]): A flag indicating if a token was randomly
                  perturbed from its original value.
                - "_transition_probs" (List): The transition matrices of the PFAs used.
                - "_transition_symbols" (List): The symbol matrices of the PFAs used.
        """
        length = 0
        sample = []
        new_language_flags = []
        learning_level = []
        perturbation = []
        num_states = []
        num_edges = []
        vocab_size = []
        last_transition_probs, last_transition_symbols = [], []
        num_language_in_sample = 0
        while (length < self.max_sample_length) and (num_language_in_sample < self.max_num_languages_per_sample + 1):
            num_sequences_for_this_language = torch.randint(
                self.sequences_per_language_min, self.sequences_per_language_max, (1,)
            ).item()

            # Choose learning level
            current_level = self.included_learning_levels[
                torch.randint(len(self.included_learning_levels), (1,)).item()
            ]

            if current_level == 0:
                all_sequences_from_language = self.level0_sequences[
                    torch.randint(len(self.level0_sequences), (1,)).item()
                ]
                seq_idxs = torch.randint(
                    len(all_sequences_from_language), (num_sequences_for_this_language,)
                )
                language_sequences = [
                    all_sequences_from_language[idx] for idx in seq_idxs
                ]
                l_num_states, l_num_edges, l_effective_vocab_size = (0, 0, 0)
            elif current_level == 4:
                # create random sequences
                language_sequences = []
                for _ in range(num_sequences_for_this_language):
                    sequence_length = torch.randint(
                        self.seq_len_min, self.seq_len_max, (1,)
                    ).item()
                    sequence = torch.randint(
                        self.alphabet_size_max, (sequence_length,)
                    ).tolist()
                    language_sequences.append(
                        "".join([self.letters[s] for s in sequence])
                    )
                l_num_states, l_num_edges, l_effective_vocab_size = (0, 0, 0)
            elif current_level == 5:
                # create half the number of random sequences, and repeat them
                half_number_of_sequences = num_sequences_for_this_language // 2
                original_sequences = []
                for _ in range(half_number_of_sequences):
                    sequence_length = torch.randint(
                        self.seq_len_min, self.seq_len_max, (1,)
                    ).item()
                    sequence = torch.randint(
                        self.alphabet_size_max, (sequence_length,)
                    ).tolist()
                    original_sequences.append(
                        "".join([self.letters[s] for s in sequence])
                    )
                if self.shuffle_random_sequences:
                    language_sequences = original_sequences * 2
                    language_sequences = np.random.permutation(language_sequences)
                else:
                    # repeat each sequence directly after the original
                    language_sequences = []
                    for seq in original_sequences:
                        language_sequences.append(seq)
                        language_sequences.append(seq)
                l_num_states, l_num_edges, l_effective_vocab_size = (0, 0, 0)
            else:
                if current_level == 1:
                    transition_probs, transition_symbols = self.level1_languages[
                        torch.randint(len(self.level1_languages), (1,)).item()
                    ]
                elif current_level == 2:
                    transition_probs, transition_symbols = self.level2_languages[
                        torch.randint(len(self.level2_languages), (1,)).item()
                    ]
                    # change the transition symbols
                    l2_vocab_size = torch.randint(
                        self.alphabet_size_min, self.alphabet_size_max, (1,)
                    ).item()
                    vocabulary = torch.arange(self.alphabet_size_max)[
                        torch.randperm(self.alphabet_size_max)[:l2_vocab_size]
                    ]
                    for state in range(transition_symbols.shape[0]):
                        edge_exists = transition_symbols[state] >= 0
                        num_outgoing_edges = edge_exists.sum()
                        outgoing_symbols = vocabulary[
                            torch.randperm(l2_vocab_size)[:num_outgoing_edges]
                        ]
                        transition_symbols[state][edge_exists] = outgoing_symbols
                elif current_level == 3:
                    transition_probs, transition_symbols = self.generate_language()
                elif current_level == 6: # mixed memorized and learned sequences
                    

                language_sequences = []
                # compute metadata for the language
                l_num_states = transition_probs.shape[0]
                l_num_edges = (transition_symbols >= 0).sum().item()
                l_effective_vocab_size = len(
                    torch.unique(transition_symbols[transition_symbols >= 0])
                )

                if self.constrained_sequences == 0:
                    for _ in range(num_sequences_for_this_language):
                        sequence, _ = self.generate_sequence(
                            transition_probs, transition_symbols
                        )
                        language_sequences.append(sequence)
                else:
                    constrained_transition_probs = transition_probs.clone()
                    constraint_mask = torch.rand_like(constrained_transition_probs) < self.edge_constrain_ratio
                    constrained_transition_probs[constraint_mask] = 0.
                    # If there is a row with all zeros, replace it with the unconstrained row
                    for i in range(constrained_transition_probs.shape[0]):
                        if constrained_transition_probs[i].sum() == 0:
                            constrained_transition_probs[i] = transition_probs[i]
                    # normalize the rows
                    constrained_transition_probs /= constrained_transition_probs.sum(dim=1, keepdim=True)
                    for i in range(num_sequences_for_this_language):
                        if i < self.constrained_sequences:
                            sequence, _ = self.generate_sequence(
                                constrained_transition_probs, transition_symbols
                            )
                        else:
                            sequence, _ = self.generate_sequence(
                                transition_probs, transition_symbols
                            )
                        language_sequences.append(sequence)

                last_transition_probs.append(transition_probs.tolist())
                last_transition_symbols.append(transition_symbols.tolist())


            # Perturbation
            for i, seq in enumerate(language_sequences):
                if torch.rand(1) > self.word_perturbation_rate:
                    perturbation.append(torch.zeros(len(seq) + 1, dtype=bool))
                    continue
                seq = list(seq)
                (perturb_idxs,) = torch.where(
                    torch.rand(len(seq)) < self.token_perturbation_rate
                )
                perturb_flag = torch.zeros(len(seq) + 1, dtype=bool)
                for pi in perturb_idxs:
                    seq[pi] = self.letters[
                        torch.randint(self.alphabet_size_max, (1,)).item()
                    ]
                    perturb_flag[pi] = True
                language_sequences[i] = "".join(seq)
                perturbation.append(perturb_flag)

            language_sequences_string = " ".join(language_sequences) + " "
            sample.append(language_sequences_string)
            length += len(language_sequences_string)

            new_language = torch.zeros(len(language_sequences_string), dtype=torch.bool)
            new_language[0] = True
            new_language_flags.append(new_language)

            learning_level.append(torch.ones_like(new_language) * current_level)
            num_states.append(torch.ones_like(new_language) * l_num_states)
            num_edges.append(torch.ones_like(new_language) * l_num_edges)
            vocab_size.append(torch.ones_like(new_language) * l_effective_vocab_size)

            num_language_in_sample += 1

        sample = "".join(sample[:-1])
        # remove last language
        new_language_flags = new_language_flags[:-1]
        learning_level = learning_level[:-1]
        num_states = num_states[:-1]
        num_edges = num_edges[:-1]
        vocab_size = vocab_size[:-1]
        last_transition_probs = last_transition_probs[:-1]
        last_transition_symbols = last_transition_symbols[:-1]

        if self.tokenizer is None:
            tokens = sample
            labels = sample
        else:
            # Tokenize (BOS and EOS tokens will be added)
            tokens = self.tokenizer.encode(sample)
            if self.tokenizer.max_seq_len is not None:
                tokens = truncate(tokens, self.tokenizer.max_seq_len - 1)
            # Labels are identical to tokens, the shift for autoregressive modelling happens in the _loss_step() function
            # in the recipe
            labels = tokens.copy()

        # add BOS and EOS information
        new_language_flags = (
            [False] + torch.cat(new_language_flags).tolist() + [False]
        )
        learning_level = [-1] + torch.cat(learning_level).tolist() + [-1]
        num_states = [-1] + torch.cat(num_states).tolist() + [-1]
        num_edges = [-1] + torch.cat(num_edges).tolist() + [-1]
        vocab_size = [-1] + torch.cat(vocab_size).tolist() + [-1]
        perturbation = (
            [False]
            + torch.cat(perturbation)[: len(learning_level) - 2].tolist()
            + [False]
        )

        return_dict = {
            "tokens": tokens,
            "labels": labels,
            "new_language": new_language_flags,
            "learning_level": learning_level,
            "num_states": num_states,
            "num_edges": num_edges,
            "vocab_size": vocab_size,
            "perturbation": perturbation,
            "_transition_probs": last_transition_probs,
            "_transition_symbols": last_transition_symbols,
        }

        return return_dict


In [6]:
cfg_dataset = cfg.dataset
packed_on_the_fly = cfg_dataset.pop("packed_on_the_fly", False)
packed_sequence_length = cfg_dataset.pop("packed_sequence_length", 2048)
split_across_pack = cfg_dataset.pop("split_across_pack", False)
num_workers = cfg_dataset.pop("num_workers", 8)
cfg.dataset.pop('_component_')
kwargs = cfg_dataset
ds = learning_levels_pfa_dataset(tokenizer, **kwargs)
sample = ds[0]

In [7]:
print(sample.keys())

dict_keys(['tokens', 'labels', 'new_language', 'learning_level', 'num_states', 'num_edges', 'vocab_size', 'perturbation', '_transition_probs', '_transition_symbols'])


In [8]:
packed_ds = PackedOnTheFlyDataset(
                    ds,
                    max_seq_len=packed_sequence_length,
                    padding_idx=tokenizer.pad_id,
                    world_size=1,
                    rank=0,
                    permute_indices=True,
                    split_across_pack=False,)

In [9]:
dataloader = DataLoader(
    dataset=packed_ds,
    batch_size=1,
    num_workers=0,
    worker_init_fn=packed_ds._worker_init_fn,
    collate_fn=partial(
        padded_collate_packed,
    ),
)
dataloader_sample = next(iter(dataloader))

In [21]:
print(dataloader_sample.keys())
tokens = [dataloader_sample['tokens'][0].tolist(),][0]
for i in dataloader_sample['mask'][0]:
    print(i.sum())

dict_keys(['tokens', 'labels', 'input_pos', 'mask'])
tensor(1)
tensor(2)
tensor(3)
tensor(4)
tensor(5)
tensor(6)
tensor(7)
tensor(8)
tensor(9)
tensor(10)
tensor(11)
tensor(12)
tensor(13)
tensor(14)
tensor(15)
tensor(16)
tensor(17)
tensor(18)
tensor(19)
tensor(20)
tensor(21)
tensor(22)
tensor(23)
tensor(24)
tensor(25)
tensor(26)
tensor(27)
tensor(28)
tensor(29)
tensor(30)
tensor(31)
tensor(32)
tensor(33)
tensor(34)
tensor(35)
tensor(36)
tensor(37)
tensor(38)
tensor(39)
tensor(40)
tensor(41)
tensor(42)
tensor(43)
tensor(44)
tensor(45)
tensor(46)
tensor(47)
tensor(48)
tensor(49)
tensor(50)
tensor(51)
tensor(52)
tensor(53)
tensor(54)
tensor(55)
tensor(56)
tensor(57)
tensor(58)
tensor(59)
tensor(60)
tensor(61)
tensor(62)
tensor(63)
tensor(64)
tensor(65)
tensor(66)
tensor(67)
tensor(68)
tensor(69)
tensor(70)
tensor(71)
tensor(72)
tensor(73)
tensor(74)
tensor(75)
tensor(76)
tensor(77)
tensor(78)
tensor(79)
tensor(80)
tensor(81)
tensor(82)
tensor(83)
tensor(84)
tensor(85)
tensor(86)
tensor(87)

In [23]:
sample_packed_ds = next(iter(packed_ds))

In [49]:
print(sample_packed_ds.keys())
datapoint = sample_packed_ds['tokens'].tolist()
print(*datapoint)

dict_keys(['tokens', 'labels', 'input_pos', '_debug_example_idx', 'new_language', 'learning_level', 'num_states', 'num_edges', 'vocab_size', 'perturbation', '_transition_probs', '_transition_symbols', 'seq_lens'])
2 114 107 101 104 104 101 113 105 108 101 105 106 101 101 105 101 106 102 106 101 106 105 101 106 101 101 105 101 101 103 106 101 101 104 104 101 105 32 105 101 105 101 106 101 106 107 101 98 106 101 101 110 106 104 105 106 104 104 100 110 106 101 32 100 101 105 101 106 101 101 104 104 101 104 105 106 104 104 106 101 101 105 101 106 101 107 97 101 104 104 101 101 104 106 105 101 106 101 106 101 101 105 106 101 106 101 101 105 111 101 106 32 105 105 105 110 106 101 101 104 104 101 104 105 106 104 104 106 101 101 110 101 106 101 98 101 103 104 104 101 101 104 106 105 101 106 101 106 101 101 105 109 101 106 101 101 105 106 101 106 32 106 104 105 106 105 32 106 104 105 106 105 32 105 101 101 104 104 101 101 105 101 101 105 106 101 101 105 101 106 101 106 101 101 105 101 106 101 1